# Finger Health Status LSTM Classifier
**Classes**: Healthy | Moderate | Bad  
**Model**: Bidirectional LSTM  
**Features**: 22 sensor features (no gender)

In [1]:
!pip install tensorflow scikit-learn pandas numpy matplotlib seaborn joblib -q
print('Dependencies installed')

Dependencies installed



[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: C:\Python312\python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
import warnings, joblib, os, json, shutil
from collections import Counter
from datetime import datetime

warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)

print(f'TensorFlow {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
CONFIG = {
    'sequence_length': 20,
    'step_size': 5,
    'batch_size': 32,
    'epochs': 120,
    'learning_rate': 0.001,
    'n_splits': 5,
    'status_classes': ['Bad', 'Healthy', 'Moderate'],  # alphabetical for LabelEncoder
    'status_display': ['Healthy', 'Moderate', 'Bad'],
}

# Sensor-only features — no Gender, Age, Hand, Timestamp
SENSOR_COLS = [
    'Thumb', 'Index', 'Middle', 'Ring', 'Pinky',
    'AngleThumb', 'AngleIndex', 'AngleMiddle', 'AngleRing', 'AnglePinky',
    'AccelX', 'AccelY', 'AccelZ',
    'GyroX', 'GyroY', 'GyroZ',
    'AngleX', 'AngleY', 'AngleZ',
    'AbductionAngle', 'RotationAngle', 'MovementMagnitude'
]

print(f'Config: seq_len={CONFIG["sequence_length"]}, step={CONFIG["step_size"]}, features={len(SENSOR_COLS)}')
print(f'Classes: {CONFIG["status_display"]}')

In [ ]:
# ── Load Data ─────────────────────────────────────────────────────────────────
# Option A: Google Drive
from google.colab import drive
drive.mount('/content/drive')
FILE_PATH = '/content/drive/MyDrive/Finger_DataSet_modified_capitalized.csv'

# Option B: Upload directly (uncomment below and comment Option A)
# from google.colab import files
# uploaded = files.upload()
# FILE_PATH = list(uploaded.keys())[0]

df = pd.read_csv(FILE_PATH)
print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'\nStatus values: {df["Status"].value_counts().to_dict()}')
print(f'Movement values: {df["Movement"].value_counts().to_dict()}')
df.head()

In [ ]:
# ── Augmentation Functions ─────────────────────────────────────────────────────
# Moderate: reduced ROM 20-35%, mild tremor
# Bad: severely reduced ROM 45-65%, high tremor + spikes

def augment_moderate(df_src):
    """Simulate moderate hand impairment: reduced ROM + mild tremor."""
    df_mod = df_src.copy()
    df_mod['Status'] = 'Moderate'
    n = len(df_mod)
    rng = np.random.default_rng(seed=1)

    flex_cols = [c for c in ['Thumb','Index','Middle','Ring','Pinky'] if c in df_mod.columns]
    angle_f   = [c for c in ['AngleThumb','AngleIndex','AngleMiddle','AngleRing','AnglePinky'] if c in df_mod.columns]
    gyro_cols = [c for c in ['GyroX','GyroY','GyroZ'] if c in df_mod.columns]
    accel_cols= [c for c in ['AccelX','AccelY','AccelZ'] if c in df_mod.columns]

    for col in flex_cols:
        df_mod[col] = df_mod[col].values * rng.uniform(0.65, 0.80, n) + rng.normal(0, 15, n)

    for col in angle_f:
        df_mod[col] = df_mod[col].values * rng.uniform(0.65, 0.80, n) + rng.normal(0, 1.5, n)

    for col in gyro_cols:
        df_mod[col] = df_mod[col].values + rng.normal(0, 1.5, n)

    for col in accel_cols:
        df_mod[col] = df_mod[col].values + rng.normal(0, 0.05, n)

    if 'MovementMagnitude' in df_mod.columns:
        df_mod['MovementMagnitude'] = df_mod['MovementMagnitude'].values * rng.uniform(0.60, 0.80, n)

    for col in [c for c in ['AbductionAngle','RotationAngle'] if c in df_mod.columns]:
        df_mod[col] = df_mod[col].values * rng.uniform(0.70, 0.85, n)

    return df_mod


def augment_bad(df_src):
    """Simulate severe hand impairment: very restricted ROM + tremor spikes."""
    df_bad = df_src.copy()
    df_bad['Status'] = 'Bad'
    n = len(df_bad)
    rng = np.random.default_rng(seed=2)

    flex_cols = [c for c in ['Thumb','Index','Middle','Ring','Pinky'] if c in df_bad.columns]
    angle_f   = [c for c in ['AngleThumb','AngleIndex','AngleMiddle','AngleRing','AnglePinky'] if c in df_bad.columns]
    gyro_cols = [c for c in ['GyroX','GyroY','GyroZ'] if c in df_bad.columns]
    accel_cols= [c for c in ['AccelX','AccelY','AccelZ'] if c in df_bad.columns]

    for col in flex_cols:
        df_bad[col] = df_bad[col].values * rng.uniform(0.35, 0.55, n) + rng.normal(0, 30, n)

    for col in angle_f:
        df_bad[col] = df_bad[col].values * rng.uniform(0.35, 0.55, n) + rng.normal(0, 3.0, n)

    for col in gyro_cols:
        # Tremor baseline + random spikes
        tremor = rng.normal(0, 4.0, n)
        spike_mask = rng.choice([0, 1], size=n, p=[0.85, 0.15])
        spikes = spike_mask * rng.normal(0, 7, n)
        df_bad[col] = df_bad[col].values + tremor + spikes

    for col in accel_cols:
        df_bad[col] = df_bad[col].values + rng.normal(0, 0.15, n)

    if 'MovementMagnitude' in df_bad.columns:
        df_bad['MovementMagnitude'] = df_bad['MovementMagnitude'].values * rng.uniform(0.25, 0.45, n)

    for col in [c for c in ['AbductionAngle','RotationAngle'] if c in df_bad.columns]:
        df_bad[col] = df_bad[col].values * rng.uniform(0.30, 0.55, n)

    return df_bad

print('Augmentation functions ready')

In [ ]:
# ── Build Balanced 3-Class Dataset ────────────────────────────────────────────
# Remap all existing Status values → 'Healthy'
df['Status'] = 'Healthy'

n_healthy = len(df)
print(f'Healthy samples: {n_healthy}')

# Use the full healthy set as source for augmentation
df_mod = augment_moderate(df)
df_bad = augment_bad(df)

df_all = pd.concat([df, df_mod, df_bad], ignore_index=True)
df_all = df_all.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'\nBalanced dataset: {len(df_all)} rows')
dist = df_all['Status'].value_counts()
for status, count in dist.items():
    print(f'  {status:10s}: {count:6d}  ({count/len(df_all)*100:.1f}%)')

In [ ]:
# ── Feature Preparation ────────────────────────────────────────────────────────
available_cols = [c for c in SENSOR_COLS if c in df_all.columns]
missing = [c for c in SENSOR_COLS if c not in df_all.columns]
if missing:
    print(f'WARNING: missing columns: {missing}')

print(f'Using {len(available_cols)} features: {available_cols}')

X_raw = df_all[available_cols].fillna(0).values
y_raw = df_all['Status'].values

# Scale
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_raw)

# Encode — LabelEncoder sorts alphabetically: Bad=0, Healthy=1, Moderate=2
le = LabelEncoder()
y_enc = le.fit_transform(y_raw)

print(f'\nClasses: {dict(enumerate(le.classes_))}')
print(f'Feature matrix: {X_scaled.shape}')

In [ ]:
# ── Create Time Sequences ──────────────────────────────────────────────────────
def make_sequences(X, y, seq_len, step):
    seqs, labs = [], []
    for i in range(0, len(X) - seq_len, step):
        seqs.append(X[i:i + seq_len])
        # Majority label in window for temporal stability
        window_y = y[i:i + seq_len]
        labs.append(Counter(window_y).most_common(1)[0][0])
    return np.array(seqs), np.array(labs)

X_seq, y_seq = make_sequences(X_scaled, y_enc, CONFIG['sequence_length'], CONFIG['step_size'])
y_cat = keras.utils.to_categorical(y_seq, num_classes=3)

print(f'Sequence tensor: {X_seq.shape}')
print(f'Label tensor:    {y_cat.shape}')
print(f'Class distribution: {{le.classes_[k]: v for k, v in Counter(y_seq).items()}}')

X_train, X_test, y_train, y_test = train_test_split(
    X_seq, y_cat, test_size=0.2, random_state=42, stratify=y_seq
)
print(f'\nTrain: {len(X_train)}  |  Test: {len(X_test)}')

In [ ]:
# ── LSTM Model ─────────────────────────────────────────────────────────────────
def build_model(input_shape, n_classes=3):
    inp = keras.Input(shape=input_shape)

    # BiLSTM layers — bidirectional captures both past and future context
    x = layers.Bidirectional(
        layers.LSTM(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.2)
    )(inp)
    x = layers.BatchNormalization()(x)

    x = layers.Bidirectional(
        layers.LSTM(64, return_sequences=True, dropout=0.2, recurrent_dropout=0.1)
    )(x)
    x = layers.BatchNormalization()(x)

    x = layers.LSTM(32, dropout=0.2)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    # Dense head
    x = layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dropout(0.15)(x)

    out = layers.Dense(n_classes, activation='softmax')(x)

    model = keras.Model(inp, out)
    model.compile(
        optimizer=keras.optimizers.Adam(CONFIG['learning_rate']),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_model(
    input_shape=(CONFIG['sequence_length'], len(available_cols))
)
model.summary()

In [ ]:
# ── Train ──────────────────────────────────────────────────────────────────────
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=18,
        restore_best_weights=True, mode='max', verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=8,
        min_lr=1e-6, mode='min', verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        'best_finger_lstm.h5', monitor='val_accuracy',
        save_best_only=True, mode='max', verbose=0
    )
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=CONFIG['epochs'],
    batch_size=CONFIG['batch_size'],
    callbacks=callbacks,
    verbose=1
)

model.load_weights('best_finger_lstm.h5')
print('Training complete — best weights restored')

In [ ]:
# ── Evaluate ───────────────────────────────────────────────────────────────────
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Accuracy : {test_acc*100:.2f}%')
print(f'Test Loss     : {test_loss:.4f}')

y_pred_probs = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=le.classes_))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=axes[0])
axes[0].set_title(f'Confusion Matrix  (Acc: {test_acc*100:.1f}%)', fontsize=13)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Training curves
axes[1].plot(history.history['accuracy'],     label='Train')
axes[1].plot(history.history['val_accuracy'], label='Validation')
axes[1].set_title('Training Accuracy', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('finger_lstm_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── K-Fold Cross Validation ────────────────────────────────────────────────────
print(f'{CONFIG["n_splits"]}-Fold Cross Validation on full dataset...')
kfold = StratifiedKFold(n_splits=CONFIG['n_splits'], shuffle=True, random_state=42)
fold_accs = []

for fold, (tr_idx, val_idx) in enumerate(kfold.split(X_seq, y_seq), 1):
    print(f'  Fold {fold}/{CONFIG["n_splits"]} ...', end=' ', flush=True)

    X_tr, X_val = X_seq[tr_idx], X_seq[val_idx]
    y_tr  = keras.utils.to_categorical(y_seq[tr_idx],  3)
    y_val = keras.utils.to_categorical(y_seq[val_idx], 3)

    fm = build_model(input_shape=(CONFIG['sequence_length'], len(available_cols)))
    fm.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=60, batch_size=CONFIG['batch_size'],
        callbacks=[
            keras.callbacks.EarlyStopping(
                monitor='val_accuracy', patience=10,
                restore_best_weights=True, mode='max', verbose=0),
            keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss', factor=0.5, patience=5, verbose=0, mode='min')
        ],
        verbose=0
    )
    _, acc = fm.evaluate(X_val, y_val, verbose=0)
    fold_accs.append(acc)
    print(f'{acc*100:.2f}%')
    keras.backend.clear_session()

print(f'\nK-Fold Mean : {np.mean(fold_accs)*100:.2f}%')
print(f'K-Fold Std  : {np.std(fold_accs)*100:.2f}%')

In [ ]:
# ── Convert to TFLite (lightweight for Hostinger deployment) ───────────────────
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_bytes = converter.convert()

with open('finger_model.tflite', 'wb') as f:
    f.write(tflite_bytes)

print(f'TFLite model size: {len(tflite_bytes)/1024:.1f} KB')

# Verify TFLite model works
interp = tf.lite.Interpreter(model_content=tflite_bytes)
interp.allocate_tensors()
inp_det = interp.get_input_details()
out_det = interp.get_output_details()
print(f'TFLite input  shape: {inp_det[0]["shape"]}')
print(f'TFLite output shape: {out_det[0]["shape"]}')

# Quick correctness check
test_input = X_test[:1].astype(np.float32)
interp.set_tensor(inp_det[0]['index'], test_input)
interp.invoke()
tflite_pred = interp.get_tensor(out_det[0]['index'])
keras_pred  = model.predict(X_test[:1], verbose=0)
print(f'Keras  pred:  {keras_pred[0].round(3)}')
print(f'TFLite pred:  {tflite_pred[0].round(3)}')
print('TFLite verification passed')

In [ ]:
# ── Save All Artifacts ─────────────────────────────────────────────────────────
joblib.dump(scaler, 'finger_scaler.pkl')
joblib.dump(le,     'finger_label_encoder.pkl')

metadata = {
    'status_classes':  list(le.classes_),        # ['Bad', 'Healthy', 'Moderate']
    'display_labels':  {'Bad': 'Bad', 'Healthy': 'Healthy', 'Moderate': 'Moderate'},
    'sensor_features': available_cols,
    'n_features':      len(available_cols),
    'sequence_length': CONFIG['sequence_length'],
    'step_size':       CONFIG['step_size'],
    'test_accuracy':   float(test_acc),
    'kfold_mean':      float(np.mean(fold_accs)),
    'kfold_std':       float(np.std(fold_accs)),
    'created_at':      datetime.now().isoformat()
}

with open('finger_model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

# Also save full Keras model
model.save('finger_model.h5')

print('Saved files:')
for fname in ['finger_model.tflite','finger_model.h5','finger_scaler.pkl',
              'finger_label_encoder.pkl','finger_model_metadata.json']:
    size = os.path.getsize(fname) / 1024
    print(f'  {fname:<35s}  {size:7.1f} KB')

print(f'\nFinal Summary:')
print(f'  Test Accuracy  : {test_acc*100:.2f}%')
print(f'  K-Fold Mean    : {np.mean(fold_accs)*100:.2f}% ± {np.std(fold_accs)*100:.2f}%')

In [ ]:
# ── Save to Google Drive ───────────────────────────────────────────────────────
drive_path = '/content/drive/MyDrive/FingerLSTM_3Class_Deploy'
os.makedirs(drive_path, exist_ok=True)

artifacts = [
    'finger_model.tflite',
    'finger_model.h5',
    'finger_scaler.pkl',
    'finger_label_encoder.pkl',
    'finger_model_metadata.json',
    'finger_lstm_results.png',
    'best_finger_lstm.h5'
]

for fname in artifacts:
    if os.path.exists(fname):
        shutil.copy(fname, f'{drive_path}/{fname}')
        print(f'  Saved: {fname}')

print(f'\nAll artifacts saved to Google Drive: {drive_path}')

In [ ]:
# ── Download Deployment Files ──────────────────────────────────────────────────
from google.colab import files

deploy_files = [
    'finger_model.tflite',
    'finger_scaler.pkl',
    'finger_label_encoder.pkl',
    'finger_model_metadata.json'
]

print('Downloading deployment artifacts...')
for fname in deploy_files:
    if os.path.exists(fname):
        files.download(fname)
        print(f'  Downloaded: {fname}')

print('\nUpload these 4 files to your Hostinger finger-api/ folder.')